### Getting Setup (On Google Colab)

* Begin by installing some pip packages and the java development kit.

In [ ]:
!pip install pyspark --quiet
!pip install -U -q PyDrive --quiet
!apt install openjdk-8-jdk-headless &> /dev/null

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 987.4/987.4 kB 14.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


* Then set the java environmental variable

In [ ]:
# RDD -- Resilient Distributed Datasets

In [ ]:
!apt-get update
!apt-get install openjdk-11-jdk -y

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,828 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,652 kB]
Ge

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

In [ ]:
import os
os.environ["JAVA_HOME"] = "/lib/jvm/java-11-openjdk-amd64"

In [ ]:
!pip install pyspark

* Then connect to a SparkSession, setting the spark ui port to `4050`.

In [ ]:
!sudo apt update
!sudo apt install openjdk-11-jdk -y


Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
118 packages can be upgraded. Run 'apt list --upgradable' to see them.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tre

In [ ]:
# from pyspark import SparkConf, SparkContext

# conf = SparkConf() \
#     .setAppName("films") \
#     .setMaster("local[2]")

# sc = SparkContext.getOrCreate(conf)
# print(sc)


In [1]:
from pyspark import SparkContext, SparkConf

conf = SparkConf().set('spark.ui.port', '4050').setAppName("films").setMaster("local[2]")
sc = SparkContext.getOrCreate(conf=conf)

* Then we need to install ngrok which will allow us to place our local spark ui on the web.

In [ ]:
!wget https://bin.equinox.io/c/4VmDzA7iaHb/ngrok-stable-linux-amd64.zip &> /dev/null
!unzip ngrok-stable-linux-amd64.zip &> /dev/null
get_ipython().system_raw('./ngrok http 4050 &')

* And finally we get a link our Spark UI

In [ ]:
!curl -s http://localhost:4040/api/tunnels | python3 -c \
    "import sys, json; print(json.load(sys.stdin)['tunnels'][0]['public_url'])"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/usr/lib/python3.12/json/__init__.py", line 293, in load
    return loads(fp.read(),
           ^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/json/__init__.py", line 346, in loads
    return _default_decoder.decode(s)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/json/decoder.py", line 338, in decode
    obj, end = self.raw_decode(s, idx=_w(s, 0).end())
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/json/decoder.py", line 356, in raw_decode
    raise JSONDecodeError("Expecting value", s, err.value) from None
json.decoder.JSONDecodeError: Expecting value: line 1 column 1 (char 0)


In [ ]:
import json

with open("file.json", "r") as f:
    data = f.read().strip()
    if not data:
        raise ValueError("JSON file is empty")
    data = json.loads(data)

print(data)


FileNotFoundError: [Errno 2] No such file or directory: 'file.json'

In [ ]:
!ls -lh file.json

ls: cannot access 'file.json': No such file or directory


### Looking Under the Hood

Now  let's again create an RDD from our movie records.

In [2]:
movies = ['dark knight', 'dunkirk', 'pulp fiction', 'avatar']
movies

['dark knight', 'dunkirk', 'pulp fiction', 'avatar']

In [3]:
type(movies)

list

In [4]:
movies_rdd = sc.parallelize(movies)
movies_rdd

ParallelCollectionRDD[0] at readRDDFromFile at PythonRDD.scala:298

In [5]:
movies_rdd.collect()

['dark knight', 'dunkirk', 'pulp fiction', 'avatar']

In [6]:
type(movies_rdd)

pyspark.core.rdd.RDD

In [7]:
type(movies_rdd.collect())

list

And then let's capitalize the movies, and select the movies that begin with `d`.

In [8]:
movies_rdd.collect() #action

['dark knight', 'dunkirk', 'pulp fiction', 'avatar']

In [9]:
movies_rdd.take(3) #actions


['dark knight', 'dunkirk', 'pulp fiction']

In [10]:
movies

['dark knight', 'dunkirk', 'pulp fiction', 'avatar']

In [11]:
movies[0].title()

'Dark Knight'

In [12]:
movies

['dark knight', 'dunkirk', 'pulp fiction', 'avatar']

In [14]:
transform=lambda i:i.title()
movies_title=[transform(i) for i in movies]

In [15]:
movies_title

['Dark Knight', 'Dunkirk', 'Pulp Fiction', 'Avatar']

In [16]:
print(type(transform))

<class 'function'>


In [17]:
movies_title

['Dark Knight', 'Dunkirk', 'Pulp Fiction', 'Avatar']

In [18]:
movies_rdd.collect()

['dark knight', 'dunkirk', 'pulp fiction', 'avatar']

In [19]:
movies_title_rdd=movies_rdd.map(transform) ## transformation
movies_title_rdd

PythonRDD[3] at RDD at PythonRDD.scala:57

In [20]:
movies_title_rdd.collect()   # Collect() is one of the action

['Dark Knight', 'Dunkirk', 'Pulp Fiction', 'Avatar']

In [21]:
student_details = {"Name":["A","B","C"],
                   "Marks":[359,456,789]}
student_details

{'Name': ['A', 'B', 'C'], 'Marks': [359, 456, 789]}

In [22]:
import pandas as pd
student_details = pd.DataFrame(student_details)
student_details

,Name,Marks
0,A,359
1,B,456
2,C,789


In [24]:
def percentage(x):
  return (x/800)*100

In [25]:
percentage(400)

50.0

In [27]:
student_details["Percentage"] = student_details["Marks"].apply(percentage)
student_details

,Name,Marks,Percentage
0,A,359,44.875
1,B,456,57.000
2,C,789,98.625


In [28]:
movies_rdd.collect()

['dark knight', 'dunkirk', 'pulp fiction', 'avatar']

In [29]:
# Filtering the movies which are having the "d" at the 0th index position(means starting with "d")
movies_rdd.filter(lambda movies : movies[1]=='u').collect()

['dunkirk', 'pulp fiction']

In [ ]:
movies_rdd.map(lambda movie: movie.title()).take(3)
#transformations -- lazy transformations
## once you apply a transformation only the function is created but it is not applied
## you need an action to apply the transformation across your rdd

['Dark Knight', 'Dunkirk', 'Pulp Fiction']

In [ ]:
rdd1=movies_rdd.map(lambda movies :movies.title()).collect()

In [ ]:
rdd1

['Dark Knight', 'Dunkirk', 'Pulp Fiction', 'Avatar']

In [ ]:
type(rdd1)

list

In [ ]:
rdd2=movies_rdd.map(lambda movies : movies.title())

In [ ]:
rdd2

PythonRDD[24] at RDD at PythonRDD.scala:57

In [ ]:
rdd2.collect()

['Dark Knight', 'Dunkirk', 'Pulp Fiction', 'Avatar']

In [ ]:
type(rdd2)

pyspark.core.rdd.PipelinedRDD

In [ ]:
type(rdd1)

list

In [ ]:
rdd2.collect()

['Dark Knight', 'Dunkirk', 'Pulp Fiction', 'Avatar']

In [ ]:
movies_rdd.map(lambda movies :movies.title()).collect()

['Dark Knight', 'Dunkirk', 'Pulp Fiction', 'Avatar']

Now as we know, Spark will partition the dataset across the cores of the executors, and then map through the records in parallel, returning all of the results.

> <img src="https://github.com/jigsawlabs-student/pyspark-rdds/blob/main/parallel.png?raw=1" width="60%">

Now let's change the function so that this time, instead of returning all of the results, we just return the first result.

In [ ]:
movies_rdd.map(lambda movie: movie.title()).take(1)

['Dark Knight']

Now if we think about, this previous step, here we would not have to map through all of the steps just to return a single result.  And it turns out if we look at Spark, we can see that even though the dataset was distributed -- it only needed to perform work on a single partition to return one result.

> <img src="https://github.com/jigsawlabs-student/pyspark-rdds/blob/main/individual_task.png?raw=1" width="80%">

This ability, to see the end result that needs to be returned, and to work efficiently to only take the needed steps to return those results, is a valuable feature when working with large datasets.  And we can better see how Spark accomplishes it in the next section.

### A little experiment

If we run the code below, notice that nothing is returned.

In [ ]:
movies_rdd.map(lambda movie: movie.title())

PythonRDD[27] at RDD at PythonRDD.scala:57

In [ ]:
movies_rdd.map(lambda movie: movie.title()).collect()

['Dark Knight', 'Dunkirk', 'Pulp Fiction', 'Avatar']

And even if we chain the map and the filter methods, still nothing is returned.

In [ ]:
movies_rdd.collect()

['dark knight', 'dunkirk', 'pulp fiction', 'avatar']

In [ ]:
movies_rdd.map(lambda movie: movie.title()).filter(lambda movie: movie[0] == 'D')

PythonRDD[29] at RDD at PythonRDD.scala:57

It's only when we add a collect function on the end, will some data be returned.

In [ ]:
movies_rdd.filter(lambda movie: movie[0] == 'd').map(lambda movie: movie.title()).collect()

['Dark Knight', 'Dunkirk']

So above, nothing was returned when we ran the `map` and `filter` functions, because when we only executed those functions, Spark did not actually act on the data.  Then in the third line we finally did act on the data.  We told Spark that we want to both transform, and filter the data, and then return all of the results.  

So it's only when we called the `collect` function that Spark's driver determined the tasks to then send off to the executors and return the results.

### Transformations and Actions

So above we can see that the functions `map` and `filter` do not actually perform any work on our data.  Instead steps are only kicked off when we call the `collect` method.  

In Spark, the methods that kick off tasks and return results are called **actions** (eg. collect).  And methods like `map` and `filter` that are called **transformations**.  

1. Transformations

So we already saw that transformations include `map` and `filter`, and our transformations do not actually return results to our users.  Here's a couple other transformations.

* sample

The `sample` method allows us to take a random sample from our dataset.  

In [ ]:
movies_rdd.collect()

['dark knight', 'dunkirk', 'pulp fiction', 'avatar']

In [ ]:
movies_rdd.sample(fraction = 0.5, withReplacement = False)

PythonRDD[31] at RDD at PythonRDD.scala:57

In [ ]:
movies_rdd.sample(fraction = 0.5, withReplacement = False).collect()

['dunkirk', 'avatar']

> Notice that it does not return any data.

* distinct

In [ ]:
movies_rdd.collect()

['dark knight', 'dunkirk', 'pulp fiction', 'avatar']

In [ ]:
movies_rdd.distinct()  # distinct() is transformation (only distinct is being created
                       # but not acting on the data)

PythonRDD[53] at RDD at PythonRDD.scala:57

In [ ]:
movies_rdd.distinct().collect()   # When you use any of the action command("action()"), then only
                                  # distinct() command is going to act on the data.

['dark knight', 'avatar', 'dunkirk', 'pulp fiction']

Finally, we have already seen `map`, which provides a one to one transformation of our records, and `select` which filters our data.  In each case, our transformations do not return data to us.

2. Actions

Actions are a bit more about the end result.  So far we've learned about `collect`, which returns *all* of the results of a series of transformations.  

* Take

We've also seen `take`, which limits our results to a subset.

In [ ]:
movies_rdd.distinct().collect()

['dark knight', 'avatar', 'dunkirk', 'pulp fiction']

In [ ]:
movies_rdd.distinct().take(2)
# take() is a action command, distinct() is a transformation command.
# distinct() just creates a function
# take(2) is acting on the first two value of the data movie_rdd.

['dark knight', 'avatar']

> So `take` is similar to the `LIMIT` function in SQL. Notice that here our records are returned.

* Count

In [ ]:
movies_rdd.distinct().count()

4

Count simply counts the results.